# Bank Customer Churn Prediction

**Goal:** Predict which customers will churn so retention can act.

**Data:** 10,000 rows · 12 cols · Target: `churn`

**Method:** One-hot encoding, train/test split, scaling, models (LogReg/RF/XGBoost), threshold tuning for higher recall.

**Results (Test):**
- Accuracy: **…**
- Precision: **…**
- Recall: **…** (threshold = **…**)
- F1: **…**
- ROC-AUC: **…**

**Artifacts:** `best_model.pkl`, `scaler.pkl`, `model_scores.csv`


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

PATH = "/kaggle/input/bank-customer-churn-dataset/Bank Customer Churn Prediction.csv"
df = pd.read_csv(PATH)
df.columns = df.columns.str.strip()  # to clean names
print(df.shape)
df.head()


In [ ]:
df.info()
print("Churn ratio:", df['churn'].value_counts(normalize=True).round(3).to_dict())
df.describe(include='all').T

**Dropping ID**

In [ ]:
if 'customer_id' in df.columns:
    df = df.drop(columns=['customer_id'])
df.head(3)

**Encoding categories**

In [ ]:
print("Before:", df.columns.tolist())
df = pd.get_dummies(df, columns=[c for c in ['country','gender'] if c in df.columns],
                    drop_first=True)
print("After :", df.columns.tolist())
df.head(3)

**Split feature**

In [ ]:
X = df.drop(columns=['churn'])
y = df['churn']
X.shape, y.shape

**Train-test split**

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train.shape, X_test.shape

**Scaling numeric feature**

In [ ]:
from sklearn.preprocessing import StandardScaler
num_cols = [c for c in ['credit_score','age','tenure','balance','products_number','estimated_salary']
            if c in X.columns]
scaler = StandardScaler()
X_train.loc[:, num_cols] = scaler.fit_transform(X_train[num_cols])
X_test.loc[:,  num_cols] = scaler.transform(X_test[num_cols])
X_train.head(3)

**Training Baseline Model**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "LogReg": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "RF":     RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1),
    "XGB":    XGBClassifier(
        n_estimators=500, max_depth=4, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, eval_metric="logloss",
        random_state=42, n_jobs=-1
    ),
}

for name, m in models.items():
    m.fit(X_train, y_train)
    print(name, "train score:", m.score(X_train, y_train), "| test score:", m.score(X_test, y_test))


**Picking a model and get prediction**

In [ ]:
best_name = "XGB"
best_model = models[best_name]

from sklearn.metrics import roc_auc_score
y_proba = best_model.predict_proba(X_test)[:,1]
y_pred  = (y_proba >= 0.5).astype(int)
print("Model:", best_name, "| ROC-AUC:", roc_auc_score(y_test, y_proba))

**Confusion Matrix and classification report(Full evaluation)**

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))

**Plotting**

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title(f"Confusion Matrix - {best_name}")
plt.show()

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title(f"ROC Curve - {best_name}")
plt.show()

In [ ]:
if hasattr(best_model, "feature_importances_"):
    imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False).head(20)
    imp.plot(kind="barh", figsize=(6,8)); plt.gca().invert_yaxis()
    plt.title("Top 20 Feature Importances"); plt.show()
else:
    # e.g., Logistic Regression
    import numpy as np
    coefs = pd.Series(best_model.coef_[0], index=X.columns).sort_values()
    coefs.tail(15).plot(kind="barh", figsize=(6,8), title="Top Positive Coefficients"); plt.show()
    (-coefs.head(15)).plot(kind="barh", figsize=(6,8), title="Top Negative Coefficients"); plt.show()


**Choosing threshold to target recall**

In [ ]:
from sklearn.metrics import precision_recall_curve, accuracy_score, precision_score, recall_score, f1_score
prec, rec, thr = precision_recall_curve(y_test, y_proba)
target_recall = 0.70
idx = np.argmin(np.abs(rec - target_recall))
thr_chosen = thr[max(idx-1,0)] if len(thr) else 0.5

y_pred_t = (y_proba >= thr_chosen).astype(int)
print(f"Chosen threshold for ~{int(target_recall*100)}% recall: {thr_chosen:.3f}")
print("Acc/Prec/Rec/F1:",
      round(accuracy_score(y_test, y_pred_t),3),
      round(precision_score(y_test, y_pred_t),3),
      round(recall_score(y_test, y_pred_t),3),
      round(f1_score(y_test, y_pred_t),3))


**Saving artifacts**

In [ ]:
import joblib, os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

os.makedirs("/kaggle/working/artifacts", exist_ok=True)
joblib.dump(best_model, "/kaggle/working/artifacts/best_model.pkl")
joblib.dump(scaler, "/kaggle/working/artifacts/scaler.pkl")

# quick score summary
summary = pd.DataFrame([{
    "model": best_name,
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba)
}])
summary.to_csv("/kaggle/working/artifacts/model_scores.csv", index=False)
print("Saved to /kaggle/working/artifacts/")


In [ ]:
# --- Save Confusion Matrix and ROC Curve to Output (/kaggle/working/artifacts) ---
import os, matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

# Make sure the folder exists
os.makedirs("/kaggle/working/artifacts", exist_ok=True)

# Get probabilities and labels at your chosen threshold
y_proba = best_model.predict_proba(X_test)[:, 1]
thr = 0.269  # <- use your chosen threshold
y_pred = (y_proba >= thr).astype(int)

# Confusion Matrix
fig_cm, ax = plt.subplots()
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
ax.set_title(f"Confusion Matrix (thr={thr:.3f})")
fig_cm.savefig("/kaggle/working/artifacts/confusion_matrix.png", dpi=150)

# ROC Curve
fig_roc, ax = plt.subplots()
RocCurveDisplay.from_predictions(y_test, y_proba, ax=ax)
ax.set_title("ROC Curve")
fig_roc.savefig("/kaggle/working/artifacts/roc_curve.png", dpi=150)

print("Saved: /kaggle/working/artifacts/confusion_matrix.png and roc_curve.png")
